# Notebook 01: Chuẩn bị Dữ liệu, BPE Tokenizer & PhoBERT Token Cache
**Dự án:** Vietnamese ASR → English Translation trên Kaggle T4 ×2 (Dual-GPU DDP)

### Mục tiêu:
1. Kiểm tra môi trường Kaggle (Python >= 3.10, PyTorch, GPU T4).
2. Giải nén mã nguồn dự án `vi_asr_project.zip` vào `/kaggle/working/vi_asr_project`.
3. Tải/Stream tập dữ liệu `nguyendv02/ViMD_Dataset` (revision: `3a5b3015...`), trích xuất manifests train/valid/test chuẩn.
4. Huấn luyện SentencePiece BPE tokenizer (~1,024 vocab) **chỉ trên tập train**.
5. Tạo PhoBERT offline token cache (`[N, 768]` float16 sharded npy) với sliding-window non-truncating stitching.
6. Kiểm toán dữ liệu và rà soát data leakage (Gate G1).

In [ ]:
# 1. Chẩn đoán môi trường Kaggle
!nvidia-smi
!python -c "import torch; print('PyTorch:', torch.__version__, '| CUDA available:', torch.cuda.is_available(), '| Device count:', torch.cuda.device_count())"

In [ ]:
# 2. Nạp mã nguồn vi_asr_project (từ Zip hoặc Git Clone trực tiếp)
%cd /kaggle/working
# Cấu hình thư mục cache Hugging Face ghi vào /tmp để không bị lỗi Read-only /root/.cache
%env HF_HOME=/tmp/huggingface
%env HF_DATASETS_CACHE=/tmp/huggingface/datasets
%env TRANSFORMERS_CACHE=/tmp/huggingface/hub
%env TORCH_HOME=/tmp/torch
import os, zipfile
from pathlib import Path

work_dir = Path("/kaggle/working/vi_asr_project")
zip_source = Path("/kaggle/input/vi-asr-code/vi_asr_project.zip")
if not zip_source.exists():
    zip_source = Path("vi_asr_project.zip")

if zip_source.exists():
    with zipfile.ZipFile(zip_source, 'r') as zf:
        zf.extractall("/kaggle/working")
    print("Unpacked vi_asr_project from zip successfully.")
else:
    print("Cloning latest repository directly from GitHub...")
    !rm -rf /kaggle/working/vi_asr_project
    !git clone --recurse-submodules https://github.com/DongTienDinh/vi-asr-translate.git /kaggle/working/vi_asr_project

# Cài đặt package vi-asr ở chế độ editable
%cd /kaggle/working/vi_asr_project
!pip install -q -e .
import vi_asr
print('vi_asr successfully installed from:', vi_asr.__file__)


In [ ]:
# 3. Chuẩn bị manifests dữ liệu từ Hugging Face
!python -m vi_asr prepare-data \
    --config configs/base.yaml \
    --paths configs/paths/kaggle_prepare.yaml


In [ ]:
# 4. Huấn luyện SentencePiece BPE Tokenizer (Train-only)
!python -m vi_asr build-tokenizer \
    --config configs/base.yaml \
    --paths configs/paths/kaggle_prepare.yaml


In [ ]:
# 5. Sinh PhoBERT Offline Token Cache cho tập Train
!python -m vi_asr cache-teacher \
    --config configs/base.yaml \
    --paths configs/paths/kaggle_prepare.yaml \
    --split train \
    --device cuda:0


In [ ]:
# 6. Kiểm toán toàn diện phân chia dữ liệu và rà soát Leakage (Gate G1)
!python -m vi_asr audit \
    --config configs/base.yaml \
    --paths configs/paths/kaggle_prepare.yaml \
    --report-dir artifacts/audit_report
